In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedGroupKFold, GridSearchCV, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    precision_score, recall_score, average_precision_score
)
from xgboost import XGBClassifier

# ========= 1. 讀資料 =========
df = pd.read_csv("final_dataset_for_ml_FULL.csv", encoding="utf-8-sig")

# ========= 1.5 反轉反向指標 =========
# 原本分數越高 = 越不漂綠 / 風險越低
# 轉成分數越高 = 越漂綠 / 風險越高
reverse_cols = [
    "llama_vagueness_score_1",
    "llama_deflection_score_1"
]

for col in reverse_cols:
    if col not in df.columns:
        raise ValueError(f"{col} 不存在於資料中，無法反轉")
    df[col] = 1 - df[col]

# ========= 2. target / groups =========
if "label" not in df.columns:
    raise ValueError("label 不存在於資料中")
if "Company" not in df.columns:
    raise ValueError("Company 不存在於資料中")

y = df["label"].astype(int)
groups_all = df["Company"]

# ========= 3. feature groups =========
semantic_cols = [
    "llama_specificity_score_1",
    "llama_evidence_substantiation_score_1",
    "llama_vagueness_score_1",
    "llama_commitment_score_1",
    "llama_temporal_credibility_score_1",
    "llama_deflection_score_1",
    "llama_comparability_score_1"
]

lexical_cols = [
    "llama_has_scope_1",
    "llama_has_sbti_1",
    "llama_has_material_1",
    "llama_has_kpi_1",
    "llama_has_percent_1"
]

financial_cols = [
    "SIZE",
    "ROA",
    "Leverage",
    "Cash_flow",
    "market_cap"
]

# ========= 4. 檢查欄位 =========
all_needed_cols = semantic_cols + lexical_cols + financial_cols + ["label", "Company"]
missing_cols = [col for col in all_needed_cols if col not in df.columns]
if missing_cols:
    raise ValueError(f"以下欄位不存在於資料中: {missing_cols}")

# ========= 5. Ablation sets =========
feature_sets = {
    "M1: Semantic": semantic_cols,
    "M2: Lexical": lexical_cols,
    "M3: Financial": financial_cols,
    "M4: Semantic + Lexical": semantic_cols + lexical_cols,
    "M5: Semantic + Financial": semantic_cols + financial_cols,
    "M6: Semantic + Lexical + Financial": semantic_cols + lexical_cols + financial_cols
}

# ========= 6. 類別不平衡 =========
pos = np.sum(y == 1)
neg = np.sum(y == 0)
scale_pos_weight = neg / pos if pos > 0 else 1

print(f"Positive class count: {pos}")
print(f"Negative class count: {neg}")
print(f"scale_pos_weight: {scale_pos_weight:.4f}")

# ========= 7. outer / inner CV (group-aware) =========
outer_cv = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=42)
inner_cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

# ========= 8. XGBoost pipeline =========
pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        scale_pos_weight=scale_pos_weight,
        random_state=42,
        n_jobs=-1
    ))
])

# ========= 9. GridSearchCV 參數 =========
param_grid = {
    "model__n_estimators": [100, 300],
    "model__max_depth": [3, 5],
    "model__learning_rate": [0.05, 0.1],
    "model__subsample": [0.8],
    "model__colsample_bytree": [0.8]
}

# ========= 10. 找最佳 threshold =========
def find_best_threshold(y_true, y_prob):
    thresholds = np.arange(0.10, 0.91, 0.01)
    best_threshold = 0.50
    best_score = -1

    for t in thresholds:
        y_pred = (y_prob >= t).astype(int)
        score = f1_score(y_true, y_pred, zero_division=0)
        if score > best_score:
            best_score = score
            best_threshold = t

    return best_threshold, best_score

# ========= 11. 評估函數 =========
def evaluate_with_grouped_nested_cv_and_oof_threshold(X, y, groups, feature_name):
    fold_metrics = []
    best_params_list = []
    best_thresholds = []

    print(f"\n========== {feature_name} ==========")
    print(f"Num features: {X.shape[1]}")

    for fold_idx, (train_idx, test_idx) in enumerate(
        outer_cv.split(X, y, groups=groups), start=1
    ):
        print(f"[{feature_name}] Fold {fold_idx} started")

        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        groups_train = groups.iloc[train_idx]

        # inner CV: 找最佳模型參數
        grid = GridSearchCV(
            estimator=pipe,
            param_grid=param_grid,
            cv=inner_cv,
            scoring="f1",
            n_jobs=-1,
            refit=True
        )
        grid.fit(X_train, y_train, **{"groups": groups_train})

        best_model = grid.best_estimator_
        best_params_list.append(grid.best_params_)

        # 用 outer train 的 OOF probabilities 找最佳 threshold
        oof_prob = cross_val_predict(
            estimator=best_model,
            X=X_train,
            y=y_train,
            groups=groups_train,
            cv=inner_cv,
            method="predict_proba",
            n_jobs=-1
        )[:, 1]

        best_threshold, best_f1 = find_best_threshold(y_train, oof_prob)
        best_thresholds.append(best_threshold)

        # 用最佳模型重訓 outer training fold
        best_model.fit(X_train, y_train)

        # outer test 評估
        test_prob = best_model.predict_proba(X_test)[:, 1]
        y_pred = (test_prob >= best_threshold).astype(int)

        fold_result = {
            "accuracy": accuracy_score(y_test, y_pred),
            "f1": f1_score(y_test, y_pred, zero_division=0),
            "roc_auc": roc_auc_score(y_test, test_prob),
            "precision": precision_score(y_test, y_pred, zero_division=0),
            "recall": recall_score(y_test, y_pred, zero_division=0),
            "average_precision": average_precision_score(y_test, test_prob)
        }
        fold_metrics.append(fold_result)

        print(
            f"[{feature_name}] Fold {fold_idx} done | "
            f"Threshold={best_threshold:.2f} | "
            f"OOF_F1={best_f1:.4f} | "
            f"Test_F1={fold_result['f1']:.4f}"
        )
        print(f"[{feature_name}] Best params: {grid.best_params_}")

    return {
        "Model": "XGBoost",
        "Feature_Set": feature_name,
        "Num_Features": X.shape[1],

        "Accuracy_mean": np.mean([m["accuracy"] for m in fold_metrics]),
        "F1_mean": np.mean([m["f1"] for m in fold_metrics]),
        "ROC_AUC_mean": np.mean([m["roc_auc"] for m in fold_metrics]),
        "Precision_mean": np.mean([m["precision"] for m in fold_metrics]),
        "Recall_mean": np.mean([m["recall"] for m in fold_metrics]),
        "PR_AUC_mean": np.mean([m["average_precision"] for m in fold_metrics]),

        "Accuracy_std": np.std([m["accuracy"] for m in fold_metrics], ddof=1),
        "F1_std": np.std([m["f1"] for m in fold_metrics], ddof=1),
        "ROC_AUC_std": np.std([m["roc_auc"] for m in fold_metrics], ddof=1),
        "Precision_std": np.std([m["precision"] for m in fold_metrics], ddof=1),
        "Recall_std": np.std([m["recall"] for m in fold_metrics], ddof=1),
        "PR_AUC_std": np.std([m["average_precision"] for m in fold_metrics], ddof=1),

        "Mean_Best_Threshold": np.mean(best_thresholds),
        "Best_Params_Per_Fold": str(best_params_list)
    }

# ========= 12. 執行 =========
results = []

for feature_name, cols in feature_sets.items():
    X = df[cols].copy()
    results.append(
        evaluate_with_grouped_nested_cv_and_oof_threshold(
            X, y, groups_all, feature_name
        )
    )

results_df = pd.DataFrame(results)

# ========= 13. 四捨五入 =========
numeric_cols = results_df.select_dtypes(include=[np.number]).columns
results_df[numeric_cols] = results_df[numeric_cols].round(4)

print("\n===== Final Results =====")
print(results_df)

# ========= 14. 存檔 =========
output_file = "llama_XGB_grouped_nestedCV_oof_threshold_with_std.csv"
results_df.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)

print(f"\nResults saved to: {output_file}")

Positive class count: 32
Negative class count: 296
scale_pos_weight: 9.2500

========== M1: Semantic ==========
Num features: 7
[M1: Semantic] Fold 1 started
[M1: Semantic] Fold 1 done | Threshold=0.63 | OOF_F1=0.5753 | Test_F1=0.7273
[M1: Semantic] Best params: {'model__colsample_bytree': 0.8, 'model__learning_rate': 0.05, 'model__max_depth': 3, 'model__n_estimators': 100, 'model__subsample': 0.8}
[M1: Semantic] Fold 2 started
[M1: Semantic] Fold 2 done | Threshold=0.13 | OOF_F1=0.5319 | Test_F1=0.5714
[M1: Semantic] Best params: {'model__colsample_bytree': 0.8, 'model__learning_rate': 0.1, 'model__max_depth': 5, 'model__n_estimators': 300, 'model__subsample': 0.8}
[M1: Semantic] Fold 3 started
[M1: Semantic] Fold 3 done | Threshold=0.50 | OOF_F1=0.5714 | Test_F1=0.8571
[M1: Semantic] Best params: {'model__colsample_bytree': 0.8, 'model__learning_rate': 0.05, 'model__max_depth': 3, 'model__n_estimators': 100, 'model__subsample': 0.8}
[M1: Semantic] Fold 4 started
[M1: Semantic] Fold 4

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedGroupKFold, GridSearchCV, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    precision_score, recall_score, average_precision_score
)
from xgboost import XGBClassifier

# ========= 1. 讀資料 =========
df = pd.read_csv("final_dataset_for_ml_FULL.csv", encoding="utf-8-sig")

# ========= 1.5 反轉反向指標 =========
# 原本分數越高 = 越不漂綠 / 風險越低
# 轉成分數越高 = 越漂綠 / 風險越高
reverse_cols = [
    "llama_vagueness_score_1",
    "llama_deflection_score_1"
]

for col in reverse_cols:
    if col not in df.columns:
        raise ValueError(f"{col} 不存在於資料中，無法反轉")
    df[col] = 1 - df[col]

# ========= 2. target / groups =========
if "label" not in df.columns:
    raise ValueError("label 不存在於資料中")
if "Company" not in df.columns:
    raise ValueError("Company 不存在於資料中")

y = df["label"].astype(int)
groups_all = df["Company"]

# ========= 3. feature groups =========
semantic_cols = [
    "chatgpt_specificity_score_1",
    "chatgpt_evidence_substantiation_score_1",
    "chatgpt_vagueness_score_1",
    "chatgpt_commitment_score_1",
    "chatgpt_temporal_credibility_score_1",
    "chatgpt_deflection_score_1",
    "chatgpt_comparability_score_1"
]

lexical_cols = [
    "chatgpt_has_scope_1",
    "chatgpt_has_sbti_1",
    "chatgpt_has_material_1",
    "chatgpt_has_kpi_1",
    "chatgpt_has_percent_1"
]

financial_cols = [
    "SIZE",
    "ROA",
    "Leverage",
    "Cash_flow",
    "market_cap"
]

# ========= 4. 檢查欄位 =========
all_needed_cols = semantic_cols + lexical_cols + financial_cols + ["label", "Company"]
missing_cols = [col for col in all_needed_cols if col not in df.columns]
if missing_cols:
    raise ValueError(f"以下欄位不存在於資料中: {missing_cols}")

# ========= 5. Ablation sets =========
feature_sets = {
    "M1: Semantic": semantic_cols,
    "M2: Lexical": lexical_cols,
    "M3: Financial": financial_cols,
    "M4: Semantic + Lexical": semantic_cols + lexical_cols,
    "M5: Semantic + Financial": semantic_cols + financial_cols,
    "M6: Semantic + Lexical + Financial": semantic_cols + lexical_cols + financial_cols
}

# ========= 6. 類別不平衡 =========
pos = np.sum(y == 1)
neg = np.sum(y == 0)
scale_pos_weight = neg / pos if pos > 0 else 1

print(f"Positive class count: {pos}")
print(f"Negative class count: {neg}")
print(f"scale_pos_weight: {scale_pos_weight:.4f}")

# ========= 7. outer / inner CV (group-aware) =========
outer_cv = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=42)
inner_cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

# ========= 8. XGBoost pipeline =========
pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        scale_pos_weight=scale_pos_weight,
        random_state=42,
        n_jobs=-1
    ))
])

# ========= 9. GridSearchCV 參數 =========
param_grid = {
    "model__n_estimators": [100, 300],
    "model__max_depth": [3, 5],
    "model__learning_rate": [0.05, 0.1],
    "model__subsample": [0.8],
    "model__colsample_bytree": [0.8]
}

# ========= 10. 找最佳 threshold =========
def find_best_threshold(y_true, y_prob):
    thresholds = np.arange(0.10, 0.91, 0.01)
    best_threshold = 0.50
    best_score = -1

    for t in thresholds:
        y_pred = (y_prob >= t).astype(int)
        score = f1_score(y_true, y_pred, zero_division=0)
        if score > best_score:
            best_score = score
            best_threshold = t

    return best_threshold, best_score

# ========= 11. 評估函數 =========
def evaluate_with_grouped_nested_cv_and_oof_threshold(X, y, groups, feature_name):
    fold_metrics = []
    best_params_list = []
    best_thresholds = []

    print(f"\n========== {feature_name} ==========")
    print(f"Num features: {X.shape[1]}")

    for fold_idx, (train_idx, test_idx) in enumerate(
        outer_cv.split(X, y, groups=groups), start=1
    ):
        print(f"[{feature_name}] Fold {fold_idx} started")

        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        groups_train = groups.iloc[train_idx]

        # inner CV: 找最佳模型參數
        grid = GridSearchCV(
            estimator=pipe,
            param_grid=param_grid,
            cv=inner_cv,
            scoring="f1",
            n_jobs=-1,
            refit=True
        )
        grid.fit(X_train, y_train, **{"groups": groups_train})

        best_model = grid.best_estimator_
        best_params_list.append(grid.best_params_)

        # 用 outer train 的 OOF probabilities 找最佳 threshold
        oof_prob = cross_val_predict(
            estimator=best_model,
            X=X_train,
            y=y_train,
            groups=groups_train,
            cv=inner_cv,
            method="predict_proba",
            n_jobs=-1
        )[:, 1]

        best_threshold, best_f1 = find_best_threshold(y_train, oof_prob)
        best_thresholds.append(best_threshold)

        # 用最佳模型重訓 outer training fold
        best_model.fit(X_train, y_train)

        # outer test 評估
        test_prob = best_model.predict_proba(X_test)[:, 1]
        y_pred = (test_prob >= best_threshold).astype(int)

        fold_result = {
            "accuracy": accuracy_score(y_test, y_pred),
            "f1": f1_score(y_test, y_pred, zero_division=0),
            "roc_auc": roc_auc_score(y_test, test_prob),
            "precision": precision_score(y_test, y_pred, zero_division=0),
            "recall": recall_score(y_test, y_pred, zero_division=0),
            "average_precision": average_precision_score(y_test, test_prob)
        }
        fold_metrics.append(fold_result)

        print(
            f"[{feature_name}] Fold {fold_idx} done | "
            f"Threshold={best_threshold:.2f} | "
            f"OOF_F1={best_f1:.4f} | "
            f"Test_F1={fold_result['f1']:.4f}"
        )
        print(f"[{feature_name}] Best params: {grid.best_params_}")

    return {
        "Model": "XGBoost",
        "Feature_Set": feature_name,
        "Num_Features": X.shape[1],

        "Accuracy_mean": np.mean([m["accuracy"] for m in fold_metrics]),
        "F1_mean": np.mean([m["f1"] for m in fold_metrics]),
        "ROC_AUC_mean": np.mean([m["roc_auc"] for m in fold_metrics]),
        "Precision_mean": np.mean([m["precision"] for m in fold_metrics]),
        "Recall_mean": np.mean([m["recall"] for m in fold_metrics]),
        "PR_AUC_mean": np.mean([m["average_precision"] for m in fold_metrics]),

        "Accuracy_std": np.std([m["accuracy"] for m in fold_metrics], ddof=1),
        "F1_std": np.std([m["f1"] for m in fold_metrics], ddof=1),
        "ROC_AUC_std": np.std([m["roc_auc"] for m in fold_metrics], ddof=1),
        "Precision_std": np.std([m["precision"] for m in fold_metrics], ddof=1),
        "Recall_std": np.std([m["recall"] for m in fold_metrics], ddof=1),
        "PR_AUC_std": np.std([m["average_precision"] for m in fold_metrics], ddof=1),

        "Mean_Best_Threshold": np.mean(best_thresholds),
        "Best_Params_Per_Fold": str(best_params_list)
    }

# ========= 12. 執行 =========
results = []

for feature_name, cols in feature_sets.items():
    X = df[cols].copy()
    results.append(
        evaluate_with_grouped_nested_cv_and_oof_threshold(
            X, y, groups_all, feature_name
        )
    )

results_df = pd.DataFrame(results)

# ========= 13. 四捨五入 =========
numeric_cols = results_df.select_dtypes(include=[np.number]).columns
results_df[numeric_cols] = results_df[numeric_cols].round(4)

print("\n===== Final Results =====")
print(results_df)

# ========= 14. 存檔 =========
output_file = "chatgpt_XGB_grouped_nestedCV_oof_threshold_with_std.csv"
results_df.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)

print(f"\nResults saved to: {output_file}")

Positive class count: 32
Negative class count: 296
scale_pos_weight: 9.2500

========== M1: Semantic ==========
Num features: 7
[M1: Semantic] Fold 1 started
[M1: Semantic] Fold 1 done | Threshold=0.76 | OOF_F1=0.6970 | Test_F1=0.7500
[M1: Semantic] Best params: {'model__colsample_bytree': 0.8, 'model__learning_rate': 0.05, 'model__max_depth': 3, 'model__n_estimators': 100, 'model__subsample': 0.8}
[M1: Semantic] Fold 2 started
[M1: Semantic] Fold 2 done | Threshold=0.49 | OOF_F1=0.7945 | Test_F1=0.0000
[M1: Semantic] Best params: {'model__colsample_bytree': 0.8, 'model__learning_rate': 0.1, 'model__max_depth': 3, 'model__n_estimators': 300, 'model__subsample': 0.8}
[M1: Semantic] Fold 3 started
[M1: Semantic] Fold 3 done | Threshold=0.41 | OOF_F1=0.6842 | Test_F1=0.8889
[M1: Semantic] Best params: {'model__colsample_bytree': 0.8, 'model__learning_rate': 0.05, 'model__max_depth': 3, 'model__n_estimators': 100, 'model__subsample': 0.8}
[M1: Semantic] Fold 4 started
[M1: Semantic] Fold 4